# Fixation mRNN Training Smoke Test

Fifty-epoch scratch runs for the `src`-native fixation mRNN workflow. These cells validate data loading, raw firing-rate training, region-PC training, checkpoint replay, feature-order restoration, flow-field analysis, and model diagnostic plots before launching larger indexed experiments.

In [ ]:
from pathlib import Path
import sys


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not find repo root")


repo_root = find_repo_root(Path.cwd().resolve())
src_root = repo_root / "src"
if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

repo_root

In [ ]:
from io import BytesIO

import pandas as pd
from IPython.display import Image, display

from dal_monte_2022_analysis.ephys.modeling import (
    build_fixation_mrnn_targets,
    compute_fixation_mrnn_currents,
    compute_fixation_mrnn_eigenvalues,
    compute_fixation_mrnn_flow_fields,
    derive_internal_feature_order_by_region,
    derive_internal_region_order,
    load_fixation_mrnn_config,
    replay_fixation_mrnn_run,
    settings_from_config,
    train_fixation_mrnn_scratch,
)
from dal_monte_2022_analysis.ephys.plotting import (
    FixationMRNNDiagnosticPlotSettings,
    plot_fixation_mrnn_activation_pc_timeseries,
    plot_fixation_mrnn_average_current_influence_pies,
    plot_fixation_mrnn_current_influence,
)


def display_figure(fig, *, dpi=150):
    buffer = BytesIO()
    fig.savefig(buffer, format="png", dpi=dpi, bbox_inches="tight")
    display(Image(data=buffer.getvalue()))


def hidden_units_by_region(run_settings, regions):
    if isinstance(run_settings.hidden_units, dict):
        return {region: int(run_settings.hidden_units[region]) for region in regions}
    return {region: int(run_settings.hidden_units) for region in regions}


def summarize_target_inventory(targets):
    print("Target inventory")
    print(f"conditions/input one-hot order: {targets.condition_names}")
    print(f"input tensor shape: {tuple(targets.input_tensor.shape)} (condition, time, one-hot input)")
    print(
        "timeline: "
        f"{len(targets.timeline_s_rel)} bins from "
        f"{targets.timeline_s_rel[0]:.3f}s to {targets.timeline_s_rel[-1]:.3f}s"
    )
    rows = []
    for region in targets.canonical_region_order:
        rows.append(
            {
                "region": region,
                "raw target shape": tuple(targets.raw_targets_by_region[region].shape),
                "raw units": len(targets.raw_feature_order_by_region[region]),
                "pc target shape": tuple(targets.pc_targets_by_region[region].shape),
                "pc dims": len(targets.pc_feature_order_by_region[region]),
                "pcs needed for 95%": targets.pca_metadata_by_region[
                    region
                ].n_components_required_for_threshold,
            }
        )
    display(pd.DataFrame(rows))


def summarize_training_setup(label, run_settings, targets):
    target_mode = run_settings.target_mode
    regions = tuple(run_settings.canonical_region_order)
    region_seed = (
        int(run_settings.seed)
        if run_settings.region_order_shuffle_seed is None
        else int(run_settings.region_order_shuffle_seed)
    )
    feature_seed = (
        int(run_settings.seed) + 1_000_003
        if run_settings.feature_order_shuffle_seed is None
        else int(run_settings.feature_order_shuffle_seed)
    )
    internal_regions = derive_internal_region_order(regions, seed=region_seed, shuffle=True)
    canonical_features = targets.feature_order_for_mode(target_mode)
    internal_features = derive_internal_feature_order_by_region(
        canonical_features,
        seed=feature_seed,
        shuffle=True,
    )
    target_by_region = targets.targets_for_mode(target_mode)
    hidden_by_region = hidden_units_by_region(run_settings, regions)

    print(label)
    print(f"target mode: {target_mode}")
    print(f"epochs: {run_settings.epochs}; seed: {run_settings.seed}; device: {run_settings.device}")
    print(f"model: activation={run_settings.activation}, dt={run_settings.dt}, tau={run_settings.tau}, spectral_radius={run_settings.spectral_radius}")
    print(f"canonical region order ({len(regions)}): {regions}")
    print(f"internal region order ({len(internal_regions)}): {internal_regions}")
    print(f"region shuffle seed: {region_seed}; feature shuffle seed: {feature_seed}")
    print(f"input tensor shape: {tuple(targets.input_tensor.shape)}")

    rows = []
    for region in regions:
        rows.append(
            {
                "region": region,
                "hidden units": hidden_by_region[region],
                "target shape": tuple(target_by_region[region].shape),
                "output dims": target_by_region[region].shape[-1],
                "feature order shuffled": tuple(canonical_features[region]) != tuple(internal_features[region]),
                "canonical feature snippet": ", ".join(map(str, canonical_features[region][:4])),
                "internal feature snippet": ", ".join(map(str, internal_features[region][:4])),
            }
        )
    display(pd.DataFrame(rows))


def summarize_replay(label, replay):
    checkpoint = replay["checkpoint"]
    print(label)
    print(f"trained target mode: {checkpoint['target_mode']}")
    print(f"canonical region order: {tuple(checkpoint['canonical_region_order'])}")
    print(f"internal region order: {tuple(checkpoint['internal_region_order'])}")
    print(f"input tensor shape: {tuple(checkpoint['input_tensor'].shape)}")
    print(
        "canonical replay output shapes: "
        f"{ {region: tuple(value.shape) for region, value in replay['canonical_output_by_region'].items()} }"
    )


MRNN_CFG = repo_root / "configs" / "ephys_fixation_mrnn.yaml"
cfg = load_fixation_mrnn_config(MRNN_CFG)
settings = settings_from_config(cfg)
settings.dataset_cfg_path = str(repo_root / "configs" / "dataset.yaml")
settings.device = "auto"
settings.epochs = 50
settings.hidden_units = 8
settings.checkpoint_every = 0
settings.seed = 123456
settings

In [ ]:
targets = build_fixation_mrnn_targets(
    settings.dataset_cfg_path,
    input_subdir=settings.input_subdir,
    dataframe_filename=settings.dataframe_filename,
    timeline_filename=settings.timeline_filename,
    canonical_region_order=settings.canonical_region_order,
    normalize_targets=settings.normalize_targets,
    normalization_stabilizer=settings.normalization_stabilizer,
    pca_variance_threshold=settings.pca_variance_threshold,
)

summarize_target_inventory(targets)

## Raw Firing-Rate Scratch Run

In [ ]:
raw_settings = settings_from_config(cfg)
raw_settings.dataset_cfg_path = settings.dataset_cfg_path
raw_settings.device = settings.device
raw_settings.epochs = 50
raw_settings.hidden_units = 8
raw_settings.seed = 123456
raw_settings.target_mode = "raw_fr"

summarize_training_setup("Raw firing-rate mRNN setup", raw_settings, targets)

raw_result = train_fixation_mrnn_scratch(
    raw_settings,
    scratch_id="smoke_raw_fr",
    overwrite=True,
)
raw_result["run_dir"], raw_result["history"].tail()

In [ ]:
raw_replay = replay_fixation_mrnn_run(raw_result["run_dir"], device="cpu")
summarize_replay("Raw firing-rate replay", raw_replay)

raw_current_df, raw_current_vectors = compute_fixation_mrnn_currents(raw_replay)
raw_eig_df = compute_fixation_mrnn_eigenvalues(raw_replay)
print(raw_current_df.head())
print(raw_eig_df.head())

## Raw Firing-Rate Diagnostic Plots

In [ ]:
plot_settings = FixationMRNNDiagnosticPlotSettings()
raw_pc_fig, raw_pc_axes = plot_fixation_mrnn_activation_pc_timeseries(
    raw_replay,
    settings=plot_settings,
)
display_figure(raw_pc_fig)

In [ ]:
raw_current_fig, raw_current_axes = plot_fixation_mrnn_current_influence(
    raw_replay,
    current_vectors=raw_current_vectors,
    settings=plot_settings,
)
display_figure(raw_current_fig)

In [ ]:
raw_pie_fig, raw_pie_axes = plot_fixation_mrnn_average_current_influence_pies(
    raw_replay,
    current_vectors=raw_current_vectors,
    settings=plot_settings,
)
display_figure(raw_pie_fig)

## Region-PC Scratch Run

In [ ]:
pc_settings = settings_from_config(cfg)
pc_settings.dataset_cfg_path = settings.dataset_cfg_path
pc_settings.device = settings.device
pc_settings.epochs = 50
pc_settings.hidden_units = 8
pc_settings.seed = 223456
pc_settings.target_mode = "region_pcs"

summarize_training_setup("Region-PC mRNN setup", pc_settings, targets)

pc_result = train_fixation_mrnn_scratch(
    pc_settings,
    scratch_id="smoke_region_pcs",
    overwrite=True,
)
pc_result["run_dir"], pc_result["history"].tail()

In [ ]:
pc_replay = replay_fixation_mrnn_run(pc_result["run_dir"], device="cpu")
summarize_replay("Region-PC replay", pc_replay)

flow = compute_fixation_mrnn_flow_fields(
    pc_replay,
    region="ofc",
    condition="face_interactive",
    num_points=5,
)
flow["region"], flow["condition"], len(flow["flow_fields"])

In [ ]:
pc_current_df, pc_current_vectors = compute_fixation_mrnn_currents(pc_replay)
pc_current_df.head()

## Region-PC Diagnostic Plots

In [ ]:
pc_pc_fig, pc_pc_axes = plot_fixation_mrnn_activation_pc_timeseries(
    pc_replay,
    settings=plot_settings,
)
display_figure(pc_pc_fig)

In [ ]:
pc_current_fig, pc_current_axes = plot_fixation_mrnn_current_influence(
    pc_replay,
    current_vectors=pc_current_vectors,
    settings=plot_settings,
)
display_figure(pc_current_fig)

In [ ]:
pc_pie_fig, pc_pie_axes = plot_fixation_mrnn_average_current_influence_pies(
    pc_replay,
    current_vectors=pc_current_vectors,
    settings=plot_settings,
)
display_figure(pc_pie_fig)